# Weeks 4–5 — Data Cleaning and Preparation

This notebook performs a focused, reproducible quality review of the combined CRMLS listing and sold ledgers. It converts date and numeric fields, creates timeline and geographic quality Flags, summarizes their prevalence, and documents a non-destructive remediation strategy.

**Source:** [IDX Exchange Data Analyst Internship Handbook, Weeks 4–5, pp. 8–9](./pdf/IDX_Exchange_Intern_Handbook_Final-version5.pdf#page=8).

**Success criteria**

- Both raw combined ledgers load without hard-coded machine paths.
- Required date fields become timezone-aware datetimes and numeric audit fields become numeric.
- Four production Flags are calculated consistently with `py/data_curation.py`.
- Before/after row counts, Flag frequencies, data types, numeric validity, and geographic quality are reported without deleting source records.

In [ ]:
%matplotlib inline
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "csv").exists() else Path.cwd().parent
LISTINGS_PATH = PROJECT_ROOT / "csv/listings.csv"
SOLD_PATH = PROJECT_ROOT / "csv/sold.csv"
PIPELINE_PATH = PROJECT_ROOT / "py/data_curation.py"
for path in (LISTINGS_PATH, SOLD_PATH, PIPELINE_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Required project file not found: {path}")

sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT.resolve()}")

## 1. Load the combined base datasets

The monthly files were previously consolidated into `csv/listings.csv` and `csv/sold.csv`. To keep run-all memory-safe near one million rows, this quality notebook reads a projection of the fields required by the Weeks 4–5 checks; it does not rewrite either raw ledger.

In [ ]:
DATE_COLUMNS = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]
NUMERIC_COLUMNS = [
    "ClosePrice", "OriginalListPrice", "LivingArea", "DaysOnMarket",
    "BedroomsTotal", "BathroomsTotalInteger", "Latitude", "Longitude",
]
FLAG_COLUMNS = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_coordinates_flag",
]
QUALITY_COLUMNS = ["ListingKey", *DATE_COLUMNS, *NUMERIC_COLUMNS]

def load_quality_projection(path):
    header = pd.read_csv(path, nrows=0).columns
    available = [column for column in QUALITY_COLUMNS if column in header]
    frame = pd.read_csv(path, usecols=available, low_memory=False)
    for column in QUALITY_COLUMNS:
        if column not in frame.columns:
            frame[column] = pd.NA
    return frame[QUALITY_COLUMNS]

listings_raw = load_quality_projection(LISTINGS_PATH)
sold_raw = load_quality_projection(SOLD_PATH)
before_rows = {"listings": len(listings_raw), "sold": len(sold_raw)}
display(pd.DataFrame({"Dataset": before_rows.keys(), "RowsLoaded": before_rows.values()}))

## 2. Normalize types and calculate quality Flags

The expected chronology is `ListingContractDate ≤ PurchaseContractDate ≤ CloseDate`. `negative_timeline_flag` is deliberately broader than negative DOM: it also captures any inversion within that sequence.

The handbook additionally requests implausible-coordinate detection. Alongside null, zero, and `Longitude > 0`, this notebook uses an approximate California envelope of latitude 32–42 and longitude −125 to −114. The envelope is a quality screen, not a legal boundary test.

In [ ]:
CA_LATITUDE_RANGE = (32.0, 42.0)
CA_LONGITUDE_RANGE = (-125.0, -114.0)

def prepare_quality_data(frame):
    data = frame.copy()
    for column in DATE_COLUMNS:
        data[column] = pd.to_datetime(data[column], errors="coerce", utc=True)
    for column in NUMERIC_COLUMNS:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    listing = data["ListingContractDate"]
    purchase = data["PurchaseContractDate"]
    close = data["CloseDate"]
    listing_after_close = listing.notna() & close.notna() & listing.gt(close)
    purchase_after_close = purchase.notna() & close.notna() & purchase.gt(close)
    listing_after_purchase = listing.notna() & purchase.notna() & listing.gt(purchase)

    latitude = data["Latitude"]
    longitude = data["Longitude"]
    missing_coordinates = latitude.isna() | longitude.isna()
    zero_coordinates = latitude.eq(0) | longitude.eq(0)
    positive_longitude = longitude.gt(0)  # Required California check: Longitude > 0
    outside_ca_envelope = (
        latitude.notna() & longitude.notna()
        & (~latitude.between(*CA_LATITUDE_RANGE) | ~longitude.between(*CA_LONGITUDE_RANGE))
    )

    data["listing_after_close_flag"] = listing_after_close.astype(bool)
    data["purchase_after_close_flag"] = purchase_after_close.astype(bool)
    data["negative_timeline_flag"] = (
        data["DaysOnMarket"].lt(0)
        | listing_after_close | purchase_after_close | listing_after_purchase
    ).astype(bool)
    data["invalid_coordinates_flag"] = (
        missing_coordinates | zero_coordinates | positive_longitude | outside_ca_envelope
    ).astype(bool)
    return data


In [ ]:
listings_clean = prepare_quality_data(listings_raw)
sold_clean = prepare_quality_data(sold_raw)
listings_clean["Dataset"] = "listings"
sold_clean["Dataset"] = "sold"
quality_data = pd.concat([listings_clean, sold_clean], ignore_index=True)

# Guard against Notebook/pipeline logic drift using a bounded parity check.
spec = importlib.util.spec_from_file_location("data_curation_quality_check", PIPELINE_PATH)
pipeline_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_module)
for raw_frame in (listings_raw.head(1000), sold_raw.head(1000)):
    notebook_flags = prepare_quality_data(raw_frame)[FLAG_COLUMNS].reset_index(drop=True)
    pipeline_flags = pipeline_module.add_data_quality_flags(raw_frame)[FLAG_COLUMNS].reset_index(drop=True)
    pd.testing.assert_frame_equal(notebook_flags, pipeline_flags)
print("Notebook and py/data_curation.py Flag logic match on bounded parity samples.")

## 3. Flag frequency and share report

Frequency answers “how many records are affected?” while share makes datasets of different sizes comparable. Flags may overlap, so percentages should not be added together.

In [ ]:
def markdown_table(frame):
    printable = frame.copy().astype(str)
    header = "| " + " | ".join(printable.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(printable.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in printable.to_numpy()]
    return "\n".join([header, separator, *rows])

flag_records = []
for dataset_name, group in quality_data.groupby("Dataset", sort=False):
    for flag in FLAG_COLUMNS:
        triggered = int(group[flag].sum())
        flag_records.append({
            "Dataset": dataset_name,
            "Flag": flag,
            "TriggeredRows": f"{triggered:,}",
            "ShareOfDataset": f"{group[flag].mean():.2%}",
        })
flag_summary = pd.DataFrame(flag_records)
display(Markdown(markdown_table(flag_summary)))

flag_plot = flag_summary.copy()
flag_plot["SharePct"] = flag_plot["ShareOfDataset"].str.rstrip("%").astype(float)
plt.figure(figsize=(12, 6))
sns.barplot(data=flag_plot, x="SharePct", y="Flag", hue="Dataset")
plt.xlabel("Share of dataset (%)")
plt.ylabel("")
plt.title("CRMLS data-quality Flag prevalence")
plt.tight_layout()
plt.show()

## 4. Handbook completeness checks

Beyond the four production Flags, the handbook explicitly asks for numeric validity, all four date conversions, and a geographic quality breakdown. The following tables provide those audits without introducing additional production Flag columns.

In [ ]:
numeric_rules = {
    "ClosePrice <= 0": quality_data["ClosePrice"].le(0),
    "LivingArea <= 0": quality_data["LivingArea"].le(0),
    "DaysOnMarket < 0": quality_data["DaysOnMarket"].lt(0),
    "BedroomsTotal < 0": quality_data["BedroomsTotal"].lt(0),
    "BathroomsTotalInteger < 0": quality_data["BathroomsTotalInteger"].lt(0),
}
numeric_audit = pd.DataFrame([
    {"Rule": rule, "TriggeredRows": f"{int(mask.sum()):,}", "Share": f"{mask.mean():.2%}"}
    for rule, mask in numeric_rules.items()
])
display(Markdown("### Invalid numeric values\n\n" + markdown_table(numeric_audit)))

latitude = quality_data["Latitude"]
longitude = quality_data["Longitude"]
geo_rules = {
    "Missing latitude or longitude": latitude.isna() | longitude.isna(),
    "Latitude or longitude equals zero": latitude.eq(0) | longitude.eq(0),
    "Longitude > 0": longitude.gt(0),
    "Outside approximate California envelope": (
        latitude.notna() & longitude.notna()
        & (~latitude.between(*CA_LATITUDE_RANGE) | ~longitude.between(*CA_LONGITUDE_RANGE))
    ),
}
geo_audit = pd.DataFrame([
    {"GeographicCheck": rule, "TriggeredRows": f"{int(mask.sum()):,}", "Share": f"{mask.mean():.2%}"}
    for rule, mask in geo_rules.items()
])
display(Markdown("### Geographic quality components\n\n" + markdown_table(geo_audit)))

dtype_audit = pd.DataFrame({
    "Field": [*DATE_COLUMNS, *NUMERIC_COLUMNS],
    "ResultingDtype": [str(quality_data[column].dtype) for column in [*DATE_COLUMNS, *NUMERIC_COLUMNS]],
})
display(Markdown("### Type confirmations\n\n" + markdown_table(dtype_audit)))

## 5. Business treatment strategy: Flag first, filter downstream

Flagged records are **not deleted from the curated ledger**. MLS anomalies can represent delayed status updates, legacy-system migrations, transcription errors, unusual transactions, or legitimate edge cases. Immediate deletion would erase audit evidence and could bias inventory, volume, brokerage, and geographic reporting.

The recommended operating policy is:

1. Preserve immutable raw CSVs and retain every row in the curated full ledger.
2. Attach transparent boolean Flags so each downstream consumer can choose its quality threshold.
3. Exclude timeline-invalid records only from duration metrics; exclude coordinate-invalid records only from maps and spatial joins.
4. Route high-impact anomalies—especially non-positive prices and repeated coordinate errors—to source-system review.
5. Publish a separate filtered analytical view when required, while keeping ListingKey traceability back to the raw record.

This preserves source truth while preventing known defects from contaminating specific measures.

## 6. Before/after data-quality summary

Because this phase uses non-destructive Flagging, the before and after row counts must match. “After” means typed and annotated—not silently reduced.

In [ ]:
after_frames = {"listings": listings_clean, "sold": sold_clean}
comparison_rows = []
for dataset_name, frame in after_frames.items():
    any_flag = frame[FLAG_COLUMNS].any(axis=1)
    comparison_rows.append({
        "Dataset": dataset_name,
        "RowsBefore": f"{before_rows[dataset_name]:,}",
        "RowsAfter": f"{len(frame):,}",
        "RowsRemoved": f"{before_rows[dataset_name] - len(frame):,}",
        "RowsWithAnyFlag": f"{int(any_flag.sum()):,}",
        "FlaggedShare": f"{any_flag.mean():.2%}",
    })
before_after = pd.DataFrame(comparison_rows)
display(Markdown(markdown_table(before_after)))
assert all(before_rows[name] == len(frame) for name, frame in after_frames.items())
display(Markdown(
    "**Conclusion.** Row preservation is 100%. The after-state improves analytical safety through "
    "normalized types, explicit chronology checks, and transparent geographic Flags while retaining "
    "ListingKey-level traceability to both raw ledgers."
))

## Recommended next step

Run `py/data_curation.py` to materialize the same four production Flags into `csv/listings_curated.csv` and `csv/sold_curated.csv`. For Week 6 metrics, apply metric-specific filters rather than a single global row deletion rule.